# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum / resume',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio / project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project / case study',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / insights', 'url': 'https://edwarddonner.com/posts/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'forum / community', 'url': 'https://discuss.huggingface.co'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
123k
•
2.76k
moonshotai/Kimi-K2.6
Updated
3 days ago
•
376k
•
1.04k
Qwen/Qwen3.6-27B
Updated
2 days ago
•
330k
•
837
openai/privacy-filter
Updated
4 days ago
•
35.8k
•
808
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
46k
•
706
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
173
ML Intern
🤖
173
Chat with an AI ML intern for quick technical help
Running
on
Zero
MCP
2.38k
Wan2.2 14B Preview
🐌
2.38k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
6

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n2 days ago\n•\n123k\n•\n2.76k\nmoonshotai/Kimi-K2.6\nUpdated\n3 days ago\n•\n376k\n•\n1.04k\nQwen/Qwen3.6-27B\nUpdated\n2 days ago\n•\n330k\n•\n837\nopenai/privacy-filter\nUpdated\n4 days ago\n•\n35.8k\n•\n808\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n2 days ago\n•\n46k\n•\n706\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n173\nML Intern\n🤖\n1

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI and machine learning community platform dedicated to building the future of open and ethical AI. It is the central hub where data scientists, engineers, researchers, and end users collaborate to create, share, and discover state-of-the-art machine learning models, datasets, and applications.

At the heart of the AI revolution, Hugging Face offers the most popular open-source ML libraries and tools and fosters innovation through a fast-growing global community. Their mission is to empower the next generation of machine learning professionals and enthusiasts by providing a versatile and collaborative environment to push the boundaries of technology.

---

## What Hugging Face Offers

- **Models:** Explore over 2 million machine learning models spanning text, image, video, audio, and 3D modalities. From natural language processing (NLP) to advanced computer vision, users can find and contribute to cutting-edge AI models.

- **Datasets:** Access and share more than 500,000 datasets curated for diverse AI applications.

- **Spaces:** Host and deploy AI-powered applications and demos effortlessly, with options to run on CPUs, GPUs, or powerful compute environments like ZeroGPU.

- **Collaboration Tools:** Build your ML portfolio, share your work with the community, and accelerate your projects using Hugging Face’s open-source stack.

- **Enterprise Solutions:** Secure and scalable AI platform with advanced features such as Single Sign-On (SSO), detailed audit logs, granular access control, token management, analytics dashboards, private storage, and enhanced compute options to support team and organization-wide collaboration.

---

## Company Culture

Hugging Face thrives on openness, collaboration, and innovation:

- **Community-Centric:** Hugging Face builds actively with its inclusive and engaged ML community.

- **Open and Ethical AI:** Committed to transparency and ethical practices, they encourage open sharing of knowledge and tools.

- **Talent and Science Focus:** Their dedicated science team continuously explores frontiers in AI, ensuring the platform stays at the technological cutting edge.

- **Empowering Users:** Hugging Face helps users—from interns and researchers to enterprise teams—grow their expertise and impact in machine learning.

---

## Customers & Community

Hugging Face serves a diverse, global community comprising:

- Individual machine learning engineers and researchers building and sharing models and datasets.

- AI enthusiasts and learners exploring applications and contributing to open source projects.

- Enterprises requiring advanced, secure platforms to scale AI initiatives across teams with flexible contract options.

Notable traits include a highly interactive platform where millions of models, datasets, and applications are updated frequently and collaboratively.

---

## Careers

Joining Hugging Face means becoming part of a forward-thinking, community-driven company with a focus on pioneering open AI technologies. Career opportunities range across research, engineering, product development, and community management roles.

Employees can expect:

- To work with an expert team pushing the limits of AI.

- A culture that values openness, collaboration, and ethical innovation.

- Opportunities to contribute to some of the most widely used open-source AI tools and infrastructure.

- The chance to impact AI development on a global scale, supporting a community that shapes the future of machine learning.

Explore current job openings and learn more about the company culture on their [Careers page](https://huggingface.co/careers).

---

## Get Started with Hugging Face

- **Join the community:** Sign up for free to explore millions of models and datasets, upload your own projects, and start collaborating.

- **Enterprise plans:** Scale your organization’s AI capabilities with premium security, support, and compute resources starting at $20/user/month.

- **Learn and share:** Use Hugging Face’s documentation, forums, and other resources to accelerate your machine learning journey.

Explore more at [huggingface.co](https://huggingface.co).

---

## Hugging Face Brand Colors

- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

---

Hugging Face — The AI community building the future.​

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community building the future of machine learning. It is a collaborative platform where data scientists, researchers, and developers around the world come together to create, share, and improve AI models, datasets, and applications across all modalities—text, image, video, audio, and even 3D. With over 2 million models and hundreds of thousands of datasets, Hugging Face enables accelerated innovation and democratizes access to state-of-the-art AI technologies.

---

## What We Offer

- **Models:** Access and contribute to over 2 million machine learning models across diverse AI tasks.
- **Datasets:** Discover and share from a vast repository of 500,000+ datasets.
- **Spaces:** Host and showcase machine learning applications in an easy-to-use, shareable format.
- **Buckets:** Efficient data storage to help manage large-scale AI resources.
- **Collaboration:** Tools and infrastructure to collaborate seamlessly with individuals or teams.

---

## Enterprise & Team Solutions

Scale your organization's AI capabilities with Hugging Face's robust enterprise offerings including:

- **Team Plans** starting at $20/user/month for small to growing teams.
- **Enterprise Solutions** featuring flexible contract options tailored to your needs.
- **Enterprise-grade security**: Single Sign-On (SSO), audit logs, advanced security policies.
- **Access controls**: Granular Resource Groups management and token control.
- **Analytics Dashboards**: Monitor repository usage and spending.
- **Advanced Compute**: Increased scalability with options like ZeroGPU and quota boosts.
- **Private Storage and Dataset Viewers**: Secure collaboration tools with additional storage.
- **Priority Support**: Dedicated enterprise support to maximize your AI projects.

---

## Pricing Overview

- **PRO Individual Account**: $9/month, offering enhanced storage, inference credits, priority queue access, and advanced hosting capabilities.
- **Team Accounts**: Starting at $20/user/month, designed for collaborative AI development with enterprise-grade features.
- Contact sales for custom Enterprise plans with flexible billing.

---

## Community & Collaboration

Hugging Face prides itself on a culture of openness, learning, and collaboration. Members of the community can:

- Share their work and build their professional portfolios in machine learning.
- Explore trending AI models and applications developed by peers.
- Participate in a global ecosystem that supports open source AI innovation.
- Leverage advanced tools to accelerate AI development and experimentation.
  
---

## Careers at Hugging Face

Join Hugging Face and be part of a purpose-driven company aiming to democratize AI. Hugging Face offers exciting career opportunities for:

- Machine Learning Engineers and Researchers
- Software Developers
- Data Scientists
- Product Managers
- Community Advocates

Work alongside passionate experts, contribute to cutting-edge AI projects, and help shape the future of machine learning. Explore open positions and grow your career with Hugging Face.

---

## Why Choose Hugging Face?

- **Leading AI Platform:** Trusted by the world’s foremost AI organizations and researchers.
- **Open Source Commitment:** Benefit from and contribute to a rich ecosystem of tools and libraries.
- **Multimodal AI:** Work across text, images, audio, video, and 3D in one platform.
- **Inclusive & Collaborative Community:** Engage with thousands of contributors worldwide.
- **Enterprise-Ready:** Robust security, management, and support suited for organizations of all sizes.

---

## Get Started Today

Build, share, and collaborate on the future of AI with Hugging Face.

- Visit [https://huggingface.co](https://huggingface.co)
- Sign up for free to access thousands of models and datasets.
- Explore PRO and Enterprise plans to unlock premium features.

---

**Hugging Face**  
The AI community building the future.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>